# 01 - Setup

By the end of this notebook you will have:

1. a **virtual environment** with **dbt** installed,
2. a **dbt profile** that connects to the Databricks SQL warehouse,
3. proof that everything works (`dbt debug`).

Steps 1-4 are done **in a terminal** (not in this notebook). Then you select the environment as the kernel of this notebook
and run the check cells, which tell you with ✅ / ❌ what is fine and what to fix.
Works the same on **Windows** and **macOS**; commands that differ are shown side by side.

> Full reference with troubleshooting: [SETUP.md](../../SETUP.md).

## 1. Open a terminal in the project folder

In VS Code: open the folder `dbt-training` (**File > Open Folder**), then **Terminal > New Terminal** (or `` Ctrl+` ``).
The terminal must be in the **root of the repository** (the folder that contains `requirements-dev.txt`). Check with:

| macOS / Linux | Windows (PowerShell) |
|---|---|
| `pwd` | `pwd` |

You also need **Python 3.10 - 3.13** and **Git** (see the installation table in SETUP.md). Check:

| macOS / Linux | Windows (PowerShell) |
|---|---|
| `python3 --version` | `py --version` |

## 2. Create the virtual environment

A virtual environment is a private folder (`.venv`) holding the Python packages of this project, so nothing else on your machine is affected.

**macOS / Linux**

```bash
python3 -m venv .venv
```

**Windows (PowerShell)**

```powershell
py -m venv .venv
```

Then **activate** it. Your prompt gets a `(.venv)` prefix.

**macOS / Linux**

```bash
source .venv/bin/activate
```

**Windows (PowerShell)**

```powershell
.venv\Scripts\Activate.ps1
```

> **macOS:** always write `source` in front, otherwise you get "permission denied".
> **Windows:** if PowerShell says *running scripts is disabled*, run `Set-ExecutionPolicy -Scope CurrentUser -ExecutionPolicy RemoteSigned` once and activate again
> (or use `cmd` with `.venv\Scripts\activate.bat`, or Git Bash with `source .venv/Scripts/activate`).

You have to activate the environment again **every time you open a new terminal**.

## 3. Install dbt

With the environment activated (your prompt starts with `(.venv)`; if it does not, go back to step 2):

```bash
python -m pip install --upgrade pip
python -m pip install -r requirements-dev.txt
```

This installs `dbt-core` and `dbt-databricks` (version 1.12 or later), plus what the notebooks need. It takes a few minutes. Then check:

```bash
dbt --version
```

You should see `dbt-core` and the `databricks` plugin, both **1.12 or later**.

## 4. Create your dbt profile

dbt reads the connection settings from a **profile** named `dbt-training`, stored **outside** the repository:

| macOS / Linux | Windows |
|---|---|
| `~/.dbt/profiles.yml` | `%USERPROFILE%\.dbt\profiles.yml` |

You need four values. Your trainer gives you the first two.

| Value | Example |
|---|---|
| **host** | `dbc-cc21e849-ca2a.cloud.databricks.com` (the workspace URL without `https://`) |
| **http_path** | `/sql/1.0/warehouses/c2a5ecbfd978e48c` (SQL Warehouses > your warehouse > *Connection details*) |
| **token** | a *personal access token* (below) |
| **schema** | **your personal schema**: your first and last name in lower case with an underscore, e.g. `olivier_hurni` |

**Create your token:** in Databricks click your user icon > **Settings** > **Developer** > **Access tokens** > **Manage** > **Generate new token**.
Copy it right away (it starts with `dapi`), it is shown only once. Treat it like a password: never commit it, never paste it in a chat.

**Option A - `dbt init`** (in the terminal, from the repository root, environment activated). Answer the prompts:
host, token (hidden), http_path, catalog **`bronze`**, schema **your personal schema**, threads `4`.

```bash
dbt init
```

**Option B - write the file yourself.** Create or edit `profiles.yml` (create the `.dbt` folder if needed):

```yaml
dbt-training:
  target: dev
  outputs:
    dev:
      type: databricks
      schema: olivier_hurni            # YOUR personal schema
      host: dbc-cc21e849-ca2a.cloud.databricks.com
      http_path: /sql/1.0/warehouses/c2a5ecbfd978e48c
      token: <your personal access token, starts with dapi>
      threads: 16
      catalog: bronze
```

| macOS / Linux | Windows (PowerShell) |
|---|---|
| `mkdir -p ~/.dbt && code ~/.dbt/profiles.yml` | `mkdir $env:USERPROFILE\.dbt -Force; code $env:USERPROFILE\.dbt\profiles.yml` |

> **Why a personal schema?** The bronze data is shared by the whole class, but everything *you* build goes into
> `silver.<your_schema>` and `gold.<your_schema>`, so nobody overwrites anybody else's work.

## 5. Use the environment in this notebook

Top right of this notebook click **Select Kernel** > **Python Environments** > **`.venv`**.
(Not in the list? Command palette `Ctrl/Cmd+Shift+P` > *Python: Select Interpreter* > the one in `.venv`, then *Developer: Reload Window*.)

Run the cell below. It only uses the standard library, so it works before anything else.

In [ ]:
import platform, sys
from importlib import metadata

def installed(pkg):
    try:
        return metadata.version(pkg)
    except metadata.PackageNotFoundError:
        return None

def at_least(version, minimum):
    return version is not None and tuple(int(x) for x in version.split(".")[:2]) >= minimum

def tick(label, ok, hint=""):
    print(("✅ " if ok else "❌ ") + label + ("" if ok else f"  -> {hint}"))
    return ok

print(platform.platform(), "| Python", sys.version.split()[0], "| kernel:", sys.prefix)
print()
ok = tick("Python 3.10 - 3.13", (3, 10) <= sys.version_info[:2] <= (3, 13), "install a supported Python version (see SETUP.md)")
ok &= tick("this notebook runs in a virtual environment", sys.prefix != sys.base_prefix,
           "select the .venv kernel (step 5), see above")
core, adapter = installed("dbt-core"), installed("dbt-databricks")
ok &= tick(f"dbt-core {core} (1.12 or later)", at_least(core, (1, 12)), "activate .venv in your terminal and run: python -m pip install -r requirements-dev.txt")
ok &= tick(f"dbt-databricks {adapter} (1.12 or later)", at_least(adapter, (1, 12)), "run: python -m pip install -r requirements-dev.txt")
ok &= tick("pandas and ipykernel installed", installed("pandas") is not None and installed("ipykernel") is not None,
           "run: python -m pip install -r requirements-dev.txt")
print("\nThe environment is ready." if ok else "\nFix the ❌ above, then re-run this cell.")

## 6. Check your profile and the connection

`helpers.py` (next to this notebook) reads your profile and gives us two functions: `q("...")` runs a SQL query, `dbt("...")` runs a dbt command.
The token is masked below.

In [ ]:
from helpers import *

profile_summary()

The `schema` must be **your personal schema**, not `default`. If you have to change the profile, edit the file, then **restart this kernel** (top toolbar) and run the cells again.

Now the real test: can we talk to the SQL warehouse? (The first call can take up to a minute if the warehouse is stopped.)

In [ ]:
SCHEMA = load_profile()["schema"]
check("your profile has a personal schema (not 'default')", SCHEMA not in (None, "", "default"),
      "set schema: <your_name> in your profile, then restart the kernel")
who = q("SELECT current_user() AS user, current_catalog() AS default_catalog")
check("the SQL warehouse answers", len(who) == 1, "check host, http_path and token in your profile")
who

## 7. Catalogs, bronze access and your personal schemas

The data lives in three Unity Catalog **catalogs**, one per layer: `bronze` (raw, shared, read-only for you), `silver` and `gold` (yours).
Check that you can read bronze and create your own schemas in silver and gold.

In [ ]:
catalogs = set(q("SHOW CATALOGS").iloc[:, 0])
for c in ["bronze", "silver", "gold"]:
    check(f"catalog '{c}' is visible", c in catalogs, "ask your trainer for access to this catalog")

try:
    q("SELECT 1 FROM bronze.sports_shop.sales_orders LIMIT 1")
    check("you can read bronze.sports_shop", True)
except Exception as e:
    check("you can read bronze.sports_shop", False, "ask your trainer (data not loaded, or missing permission). " + str(e)[:120])

for c in ["silver", "gold"]:
    try:
        q(f"CREATE SCHEMA IF NOT EXISTS {c}.{SCHEMA}")
        check(f"schema {c}.{SCHEMA} is ready", True)
    except Exception as e:
        check(f"schema {c}.{SCHEMA} is ready", False, "you need the CREATE SCHEMA privilege on this catalog, ask your trainer. " + str(e)[:120])

## 8. dbt can connect too

The dbt project you will build lives in `training/project` (almost empty on purpose, you fill it notebook after notebook).
It needs the `dbt_utils` package (`dbt deps`), and `dbt debug` tests the connection **from dbt itself**.

You can run both commands **in this notebook** (the next two cells) **or in the terminal**, whichever you prefer:

**Option A - in the notebook:** run the two cells below.

**Option B - in the terminal** (environment activated, same commands on Windows and macOS):

```bash
cd training/project
dbt deps
dbt debug
```

(From the repository root, `training/project` is the folder of the dbt project. Later you can go back with `cd ../..`.)

In [ ]:
dbt("deps")

In [ ]:
dbt("debug")

Expect `Connection test: [OK connection ok]` and `All checks passed!` at the end.

> You may see a warning *"Configuration paths exist in your dbt_project.yml file which do not apply to any resources"*:
> this is normal for now, the `silver` and `gold` folders do not contain models yet.

| Symptom | Likely cause |
|---|---|
| `dbt` not found in the terminal | the environment is not activated (step 2) |
| `Could not find profile named 'dbt-training'` | wrong profile name or file location (step 4) |
| `401` / `Invalid access token` | expired token, or a trailing space when pasting it |
| timeout / warehouse not running | wait a minute and retry, or ask your trainer |
| `Catalog ... does not exist` / permission denied | ask your trainer for access |

## 9. Final check

In [ ]:
ok = True
ok &= check("profile loads", bool(load_profile().get("host")))
ok &= check("personal schema set", load_profile()["schema"] not in (None, "", "default"))
ok &= check("warehouse answers", scalar("SELECT 1") == 1)
ok &= check("dbt_utils installed", (PROJECT / "dbt_packages" / "dbt_utils").exists(), "run dbt('deps')")
print("\nAll set, go to notebook 02!" if ok else "\nFix the ❌ above, then re-run this cell.")